In [ ]:
from pathlib import Path
import pandas as pd
import json
import re
JSON_NAME = "until_2026-07-23_삼성전자.json"
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PORCESSED_DIR = DATA_DIR / "processed"
JSON_PATH = RAW_DIR / JSON_NAME
# 판다스 디스플레이 설정
pd.set_option("display.max_colwidth", None)


In [ ]:
# 중첩이 깊은 json 로드하기
records = []

with open(JSON_PATH, "r", encoding="utf-8-sig") as file:
    for line in file:
        if line.strip():
            records.append(json.loads(line))
records[1]


In [ ]:
# 필요컬럼 확인  : commentID, message.title, message.message, board.stockCode, statistic.likeCount, parentId, createdAt, updatedAt
keys = list(records[1].keys())
key = keys[1]
print(key)
records[1]["message"]

In [ ]:
# 스네이크 케이스로 key 리네임한 dict 생성
rows = []
for record in records:

    row = {
        "comment_id": record.get("commentId"),
        "title": record.get("message", {}).get("title"),
        "message": record.get("message", {}).get("message"),
        "stock_code": record.get("board", {}).get("stockCode"),
        "like_count": record.get("statistic", {}).get("likeCount"),
        "parent_id": record.get("parentId"),
        "created_at": record.get("createdAt"),
        "updated_at": record.get("updatedAt"),
    }

    rows.append(row)

In [ ]:
# df변환후 정보확인
comments_df = pd.DataFrame(rows)
comments_df.info()

In [ ]:
# datetime 객체로 변환 후 KST 기준의 일반 datetime으로 통일
datetime_columns = ["created_at", "updated_at"]
comments_df[datetime_columns] = (
    comments_df[datetime_columns]
    .apply(pd.to_datetime, format="ISO8601", utc=True)
    .apply(
        lambda column:(
            column
            .dt.tz_convert("Asia/Seoul")
            .dt.floor("s")
            .dt.tz_localize(None)
        )
    )
)

comments_df[["created_at","updated_at"]].head()

In [ ]:
# 제목 내용 병합

def normalize_text(value):
    """결측값을 빈 문자열로 바꾸고 연속된 공백을 하나로 정리한다."""
    if pd.isna(value):
        return ""

    return re.sub(r"\s+", " ", str(value)).strip()


def merge_title_message(row):
    """문맥이 이어지는 title과 message를 중복 없이 하나의 문장으로 병합한다."""
    title = normalize_text(row["title"])
    message = normalize_text(row["message"])

    if not title:
        return message

    if not message:
        return title

    if title == message:
        return title

    if message.startswith(title):
        return message

    if title.startswith(message):
        return title

    return f"{title} {message}"


comments_df["text"] = comments_df.apply(
    merge_title_message,
    axis=1,
)

In [ ]:
# 문자열 길이컬럼 생성
comments_df["length"] = comments_df["text"].str.len()

In [ ]:
# 종목코드 앞의 A 제거
comments_df["stock_code"] = (
    comments_df["stock_code"]
    .astype("string")
    .str.strip()
    .str.replace(r"^A", "", regex=True)
)

In [ ]:
# 중복 id제거
comments_df = (
    comments_df
    .sort_values("updated_at")
    .drop_duplicates(subset=["comment_id"], keep="last")
    .reset_index(drop=True)
)

In [ ]:
# 결측행 제거
comments_df = (
    comments_df
    .dropna(subset=["comment_id", "stock_code", "text", "created_at"])
    .loc[lambda df: df["text"].ne("")]
    .reset_index(drop=True)
)

In [ ]:
before_count = len(comments_df)

comments_df = (
    comments_df
    .dropna(subset=["comment_id", "stock_code", "text", "created_at"])
    .loc[lambda df: df["text"].str.strip().ne("")]
    .reset_index(drop=True)
)

removed_count = before_count - len(comments_df)

print(f"전처리 전: {before_count:,}건")
print(f"제거된 행: {removed_count:,}건")
print(f"전처리 후: {len(comments_df):,}건")

In [ ]:
required_columns = [
    "comment_id",
    "stock_code",
    "text",
    "created_at",
]
print(comments_df[required_columns].isna().sum())
print("빈 text:", comments_df["text"].str.strip().eq("").sum())
print("중복 comment_id:", comments_df["comment_id"].duplicated().sum())